# Local Retrieval Mini-Project

This mini-project builds a transparent **local sparse retrieval** workflow. It is not a neural semantic-search system: it uses TF-IDF-style term weights and cosine similarity so every retrieval decision can be inspected.

That distinction matters. Production RAG systems often use dense embedding models and vector databases, but they still need the same engineering loop you practice here: choose trusted sources, rank candidates, evaluate retrieval quality, and define fallback behavior for weak matches.

## Learning Objectives

By the end of this mini-project, you should be able to:

- Build a local sparse retrieval index from a set of documents.
- Retrieve top-k candidates for support questions.
- Evaluate whether the expected topic appears in the top results.
- Implement a confidence threshold and fallback decision.
- Explain how this local workflow differs from dense semantic search.

## The Retrieval System You Are Building

The source data is a small course catalog plus support-style questions. Some questions are direct, some are ambiguous, and some are unrelated. This mix is useful because a retrieval system should be judged on more than happy-path examples.

```mermaid
graph LR
    Q["Support question"] --> IDX["Local sparse index"]
    C["Course catalog"] --> IDX
    IDX --> TOPK["Top-k lesson candidates"]
    TOPK --> EVAL["HitRate@k check"]
    TOPK --> DECIDE["Use or fallback"]
```

The local index is deliberately small and inspectable. It gives you the retrieval shape used in RAG without hiding the core decisions behind a library call.

**Sparse retrieval, not dense semantic search.**

The phrase _semantic search_ is often used for systems that compare dense neural embeddings from models such as Sentence Transformers or hosted embedding APIs. This notebook does something more transparent: it compares sparse TF-IDF-style vectors built from visible terms.

That means it can catch useful lexical overlap and weighted term matches, but it will not understand synonyms as well as a dense embedding model. Keep that limitation in mind when interpreting results.

## Import The Shared Retrieval Helpers

Notebook 03 introduced the vectorizer mechanics. Here we reuse the shared helper module so the mini-project can focus on system behavior: retrieval, evaluation, thresholds, and fallback decisions.

In [ ]:
import csv
from pathlib import Path

from retrieval_helpers import LocalRetrievalIndex, relevant_in_top_k

## Load The Evaluation Data

Before building retrieval behavior, inspect the available fields. The catalog provides documents; the support questions provide labeled examples for evaluation.

In [ ]:
catalog_path = Path("data") / "course_catalog.csv"
questions_path = Path("data") / "support_questions.csv"

with catalog_path.open(newline="", encoding="utf-8") as file:
    catalog = list(csv.DictReader(file))
with questions_path.open(newline="", encoding="utf-8") as file:
    questions = list(csv.DictReader(file))

print(f"Catalog lessons: {len(catalog)}")
print(f"Support questions: {len(questions)}")
print("Catalog columns:", list(catalog[0]))
print("Question columns:", list(questions[0]))

## Build The Local Index

The index combines each lesson's title, module, and description into searchable text. It then fits sparse term weights once and reuses them for every query.

In [ ]:
index = LocalRetrievalIndex.from_csv(catalog_path)

## Inspect Top-k Results

Top-3 often matters more than top-1 because a downstream RAG prompt may include several chunks. Inspect the candidates before trusting the system.

In [ ]:
question = questions[2]
results = index.search(question["question"], k=3)

print(f"Question: {question['question']}")
print(f"Expected topic: {question['expected_topic']}\n")
for result in results:
    print(
        f"{result.rank}. {result.title} | score={result.score:.3f} | module={result.module}"
    )

## Evaluate Retrieval Quality

`HitRate@k` is the fraction of questions with at least one relevant result in the first _k_ candidates. Here, we use a topic match in the lesson metadata as a simple proxy for relevance. Unlike hit rate, `recall@k` measures the fraction of all relevant documents retrieved; this dataset does not provide those document-level labels.

In [ ]:
evaluation_rows = []
for row in questions:
    results = index.search(row["question"], k=3)
    evaluation_rows.append(
        {
            "question_id": row["question_id"],
            "question": row["question"],
            "expected_topic": row["expected_topic"],
            "top_1": results[0].title,
            "top_1_score": round(results[0].score, 3),
            "relevant_in_top_3": relevant_in_top_k(results, row["expected_topic"], k=3),
        }
    )

hit_count = sum(1 for row in evaluation_rows if row["relevant_in_top_3"])
print(f"HitRate@3: {hit_count / len(evaluation_rows):.2f}")
failed_rows = [row for row in evaluation_rows if not row["relevant_in_top_3"]]
print(f"Failed topic matches: {len(failed_rows)} / {len(evaluation_rows)}")
for row in failed_rows[:5]:
    print(row)

## Read The Evaluation Rows

The output shows up to five questions whose expected topic is absent from top-3. Compare their wording, expected topic, top-1 title, and score to investigate the misses. A top-3 hit can still help a downstream prompt even when top-1 is wrong.

For a real system, keep a small labeled regression set and inspect failures after every retrieval change.

## Practice: Implement A Confidence Threshold

Sparse retrieval always returns nearest neighbors, even for unrelated questions. A threshold converts the top score into a product decision:

- use retrieved candidates when the score is high enough,
- fallback when the score is too weak.

Complete the function below, then compare your output with the reference solution.

In [ ]:
threshold = 0.15
queries = [
    "How do I estimate token cost?",
    "How do I search documents by meaning?",
    "How do I repair a bicycle chain?",
]

# @TODO: Implement a `retrieval_decision(query, threshold)` helper below.
# It should search the index, inspect the top score, and return a dictionary with:
# - query
# - top_title
# - top_score
# - decision, either "use retrieved candidates" or "fallback"


def retrieval_decision(query, threshold):
    pass

<details>
  <summary>Show Reference Solution</summary>

```python
def retrieval_decision(query: str, threshold: float) -> dict:
    results = index.search(query, k=3)
    top = results[0]
    decision = "use retrieved candidates" if top.score >= threshold else "fallback"
    return {
        "query": query,
        "top_title": top.title,
        "top_score": round(top.score, 3),
        "decision": decision,
    }


for query in queries:
    print(retrieval_decision(query, threshold))
```

</details>


## Tune The Threshold

A threshold that is too low may accept weak matches, while a threshold that is too high may reject relevant questions. Test several values and watch how the unrelated question and valid course questions move.

In [ ]:
if retrieval_decision(queries[0], threshold) is None:
    print(
        "Complete `retrieval_decision` in the practice cell above, then re-run this cell."
    )
else:
    for candidate_threshold in [0.10, 0.25, 0.45, 0.50]:
        print(f"\nThreshold: {candidate_threshold}")
        for query in queries:
            row = retrieval_decision(query, candidate_threshold)
            print(f"  {row['decision']:25} | {row['top_score']:.3f} | {row['query']}")

## From Local Sparse Retrieval To Production Search

| Responsibility | This notebook | Common production replacement |
| --- | --- | --- |
| Text representation | Transparent TF-IDF-style sparse vectors | Dense embeddings from Sentence Transformers or provider APIs |
| Retrieval interface | `LocalRetrievalIndex.search()` | LangChain retriever or custom service boundary |
| Fast vector search | In-memory cosine scoring | FAISS or a vector database index |
| Storage and metadata | CSV rows in `data/` | Chroma, Qdrant, or another vector database |
| Evaluation | Labeled support questions | Regression test set with retrieval metrics |

The local workflow is valuable because it makes every decision inspectable. Move to dense embeddings when synonym handling, multilingual retrieval, large corpora, persistence, or metadata filtering matter.

## Check Your Understanding

- Why does retrieval use top-k instead of only top-1?
- What does HitRate@k measure, and how does it differ from recall@k?
- Why is a fallback threshold useful?
- Why should this notebook not be described as dense semantic search?

<details>
<summary>Show Reference Answers</summary>

- Top-k is useful because the best source may not be ranked first, and downstream RAG prompts often include several candidate chunks.
- HitRate@k is the fraction of questions with at least one relevant result in the first k candidates. Recall@k measures the fraction of all relevant documents retrieved for each question.
- A fallback threshold prevents the application from treating every nearest neighbor as a valid answer source.
- The notebook uses sparse term-weight vectors, not neural dense embeddings. It demonstrates the retrieval workflow, but it does not provide the synonym handling usually associated with dense semantic search.

</details>